# CRTi: quality filtering and Scrublet

This excerpt preserves the QC plotting and per-sample Scrublet workflow used in my analysis. Repeated sample-specific blocks have been combined into a loop; no original sample labels or doublet results are included.

Run `01-preprocessing.ipynb` first. The input must contain unnormalized counts and a `Samples` column that identifies separate captures. Install `scrublet` in addition to the preprocessing dependencies. Review the histogram and automatic threshold for each capture before interpreting predicted doublets. No plots or data from this notebook have been published.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

# Run from the repository root. These local inputs and outputs are not distributed.
PRIVATE_DIR = Path("data/private/crti")
OUTPUT_DIR = Path("outputs/crti/python")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = OUTPUT_DIR
sc.settings.verbosity = 3
sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300,
                             frameon=True, vector_friendly=True, fontsize=14)
rcParams['figure.figsize'] = 8, 8

In [ ]:
adata = sc.read_h5ad(OUTPUT_DIR / 'preprocessed-counts.h5ad')

In [ ]:
sc.pl.violin(adata, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, size=0, multi_panel=True, save = "_pre-qc.png")
# sc.pl.scatter(adata, x='percent_mito', y='n_genes')
# sc.pl.scatter(adata, x='n_counts', y='n_genes')

In [ ]:
adata = adata[adata.obs['n_genes'] < 10000, :].copy()
adata = adata[adata.obs['n_genes'] > 200, :].copy()
adata = adata[adata.obs['percent_mito'] < 0.20, :].copy()
sc.pl.violin(adata, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, size=0, multi_panel=True, save='_post-qc.png')

In [ ]:
import matplotlib.pyplot as plt

cell_counts = adata.obs['Samples'].value_counts()
plt.figure(figsize=(10, 6))
cell_counts.plot(kind='bar')
plt.xticks(rotation=45, ha='right')
plt.title('Number of Cells per Sample')
plt.xlabel('Samples')
plt.ylabel('Cell Count')
plt.savefig(OUTPUT_DIR / 'cell-counts-per-sample.png', dpi=300, bbox_inches='tight')
plt.show()

## Per-capture doublet scoring

The original notebook scored each sample separately with `expected_doublet_rate=0.06`, then joined predictions back by barcode. This version preserves the same Scrublet calls and settings while iterating over sample labels supplied by the input. The expected rate must be reassessed for a different capture design.

In [ ]:
import scrublet as scr

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rc('font', size=14)
plt.rcParams['pdf.fonttype'] = 42

predictions = []
for sample_id in adata.obs['Samples'].dropna().unique():
    sample = adata[adata.obs['Samples'].isin([sample_id])].copy()
    # Pass the count matrix directly, keeping sparse input sparse.
    scrub = scr.Scrublet(sample.X, expected_doublet_rate=0.06)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(
        min_counts=2, min_cells=3,
        min_gene_variability_pctl=85, n_prin_comps=30,
    )
    scrub.plot_histogram()
    plt.show()
    sample_pred = pd.DataFrame({
        'Doublet': scrub.predicted_doublets_,
        'doublet_score': doublet_scores,
    }, index=sample.obs.index)
    predictions.append(sample_pred)

result = pd.concat(predictions).reindex(adata.obs_names)
if result['Doublet'].isna().any():
    raise ValueError('Missing doublet predictions: check sample metadata.')
adata.obs = adata.obs.join(result)
result.to_csv(OUTPUT_DIR / 'doublet-predictions.csv')

In [ ]:
adata.write(OUTPUT_DIR / 'qc-with-doublet-flags.h5ad')